# rynude Lyric 4.7 — QLoRA fine-tuning (Qwen3-1.7B) di Google Colab GRATIS

Notebook ini melatih **rynude Lyric 4.7** dari model dasar **Qwen3-1.7B** memakai QLoRA, lalu mengekspornya ke file **GGUF** yang siap dimasukkan ke aplikasi rynude.

**Prasyarat (Rp 0):**
1. Akun Google (untuk Colab).
2. Pilih runtime GPU gratis: menu **Runtime → Change runtime type → T4 GPU → Save**.
3. Satu file: **`dataset_siap-latih.jsonl`** (dari folder `training/lyric-4.6/`). Tidak perlu API key.

**Cara pakai: Runtime → Run all.** Saat sel unggah muncul, pilih `dataset_siap-latih.jsonl`.

⏱️ **Total ± 25–35 menit** (install + training 2.500 contoh + ekspor GGUF) — dibuat singkat agar SELESAI sebelum Colab gratis memutus sesi.

> ⚠️ Colab gratis memutus sesi jika idle. **Tetap di tab ini dan klik/scroll tiap beberapa menit** selama proses berjalan. Jangan tutup tab.

## 1. Pasang Unsloth (framework QLoRA tercepat)

In [ ]:
%%capture
# Unsloth: 2x lebih cepat, VRAM 1/2. Versi pip resmi untuk Colab.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade trl peft accelerate bitsandbytes

: 

## 2. Muat model dasar Qwen3-1.7B (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 1024  # cukup untuk data ini; hemat VRAM & lebih cepat

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-1.7B",
    max_seq_length = MAX_SEQ,
    load_in_4bit = True,
    dtype = None,
)
print("Model dasar termuat.")

## 3. Pasang adapter LoRA (hyperparameter dari rancangan loRA.md Bab 6.3)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 64,
    lora_dropout = 0.05,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("Adapter LoRA terpasang.")

## 4. Unggah data latih

Jalankan sel di bawah, lalu **pilih file `dataset_siap-latih.jsonl`** dari komputer Anda (satu file saja; validasi diambil otomatis 3%).

In [ ]:
import os
from datasets import load_dataset

DATA_FILE = "dataset_upgrade.jsonl"   # <- data 4.6 (di-upweight) + perbaikan baru

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print(f"Pilih/unggah {DATA_FILE} ...")
        files.upload()
    except Exception:
        print(f"Silakan drag-and-drop {DATA_FILE} ke panel file kiri, lalu jalankan ulang sel ini.")

assert os.path.exists(DATA_FILE), f"{DATA_FILE} belum terunggah/ditemukan di folder kerja!"

full = load_dataset("json", data_files={"train": DATA_FILE})["train"]

def to_text(ex):
    ex["text"] = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)
    return ex

full = full.map(to_text)

split = full.train_test_split(test_size=0.03, seed=42)
ds = {"train": split["train"], "validation": split["test"]}

# File upgrade sudah berukuran pas (~4000) & seimbang -> pakai SEMUA (jangan di-subset,
# supaya inti 4.6 yang di-upweight tidak ikut terbuang).
SUBSET = None
if SUBSET and len(ds["train"]) > SUBSET:
    ds["train"] = ds["train"].shuffle(seed=42).select(range(SUBSET))

print("train:", len(ds["train"]), "| validation:", len(ds["validation"]))
print("\nContoh 1 baris terformat:\n", ds["train"][0]["text"][:600])

## 5. Latih (completion-only: loss hanya pada jawaban asisten)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = ds["train"],
    eval_dataset = ds.get("validation"),
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = 768,                # lebih pendek -> lebih cepat & hemat VRAM
        packing = True,
        packing_strategy = "bfd",
        per_device_train_batch_size = 2, # naik ke 2 (muat di 768) -> ~2x lebih cepat
        gradient_accumulation_steps = 8, # batch efektif tetap 16
        warmup_steps = 10,
        num_train_epochs = 1,            # 1 epoch cukup; loss sudah turun cepat
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer.train()

## 6. Uji cepat (pastikan jawaban Bahasa Indonesia & waras)

In [ ]:
FastLanguageModel.for_inference(model)
for q in ["halo bang", "jelaskan apa itu machine learning secara singkat",
          "buatkan surat izin sakit"]:
    msgs = [{"role":"user","content":q}]
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                           return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.6,
                         top_p=0.95, top_k=20)
    print("Q:", q)
    print("A:", tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
    print("-"*60)

## 7. Ekspor ke GGUF (file yang dipakai aplikasi rynude)

Menghasilkan **Q8_0** (kualitas terbaik untuk 1.7B, ± 1.8 GB) yang cocok dipakai node-llama-cpp.

In [ ]:
# EKSPOR ke GGUF Q8_0 — inilah file yang dipakai aplikasi rynude.
# Unsloth otomatis: gabung (merge) LoRA ke model dasar -> konversi ke GGUF via llama.cpp.
# Proses konversi bisa 5-15 menit (build llama.cpp + kuantisasi). Sabar ya.
model.save_pretrained_gguf(
    "rynude-lyric-plus",          # nama folder output
    tokenizer,
    quantization_method = "q8_0", # kualitas terbaik untuk 1.7B (~1.8 GB)
)
print("Ekspor GGUF selesai. File ada di folder 'rynude-lyric-plus' (panel kiri).")

## 8. Unduh GGUF ke komputer Anda

In [ ]:
import os
path = None
folders_to_check = ["rynude-lyric-plus_gguf", "rynude-lyric-plus"]
for folder in folders_to_check:
    if os.path.exists(folder):
        for f in os.listdir(folder):
            if f.endswith(".gguf"):
                path = os.path.join(folder, f)
                break
    if path: break

if path:
    print("File GGUF siap di server Colab/VSCode:", path)
    try:
        from google.colab import files
        files.download(path)
    except Exception as e:
        print(f"Jika otomatis unduh tidak aktif di VSCode Colab, silakan klik kanan file '{path}' di panel explorer kiri lalu pilih 'Download'.")
else:
    print("File GGUF tidak ditemukan di folder mana pun!")


## Selesai! Langkah terakhir (di komputer Anda)

1. Rename file GGUF menjadi **`Qwen3-1.7B-Lyric-Plus-Q8_0.gguf`** dan taruh di `storage/app/models/`.
2. Daftarkan sebagai model baru di Model Hub (minta Claude Code menambahkannya ke katalog — jangan menimpa Lyric asli).
3. Ukur hasilnya: `php artisan rynude:eval <kode-model-baru>` dan bandingkan dengan baseline.

Kalau skornya belum naik, **perbaiki DATA** (tambah contoh untuk kasus yang masih gagal), bukan menambah epoch. Kualitas data = kualitas model.